In [3]:
#Import necessary libraries
import os
import json
import time
import warnings
import requests
import numpy as np
import pandas as pd
import random
from faker import Faker
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from datetime import datetime


warnings.filterwarnings('ignore')
load_dotenv()

False

In [4]:
#Load the dataset   
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
data = pd.read_csv(os.path.join(BASE_DIR, '..', 'data', 'raw', 'WA_Fn-UseC_-HR-Employee-Attrition.csv'))

In [5]:
#Initialize Faker and set seeds for reproducibility
fake = Faker()

Faker.seed(42)
random.seed(42)
np.random.seed(42)

In [7]:
# Verify required columns exist

required_columns = [
    "EmployeeNumber",
    "MonthlyIncome",
    "OverTime",
    "PerformanceRating"
]

missing = [col for col in required_columns if col not in data.columns]

if missing:
    raise ValueError(f"Missing columns in IBM dataset: {missing}")

In [8]:
# Ensure we have at least 1470 employees to generate 24 months of data for 1000 employees

data = data.head(1470).copy()


In [9]:
# Generate synthetic employee names using Faker

data["EmployeeName"] = [
    fake.name() for _ in range(len(data))
]

In [10]:
# Define years and months for the synthetic data generation


current_year = datetime.now().year

years = [current_year - 2, current_year - 1]

months = list(range(1, 13))

In [11]:
# Generate synthetic payroll data for 24 months for each employee

payroll_rows = []

for _, row in data.iterrows():

    employee_id = row["EmployeeNumber"]
    employee_name = row["EmployeeName"]

    # Initial salary from IBM dataset
    original_salary = row["MonthlyIncome"]

    overtime_flag = row["OverTime"]
    performance = row["PerformanceRating"]

In [12]:
# Generate synthetic payroll data for 24 months for each employee

payroll_rows = []

for _, row in data.iterrows():

    employee_id = row["EmployeeNumber"]
    employee_name = row["EmployeeName"]

    # Initial salary from IBM dataset
    original_salary = row["MonthlyIncome"]

    overtime_flag = row["OverTime"]
    performance = row["PerformanceRating"]

    # Salary Increment Logic
    # Salary changes every 12-18 months

    increment_month = random.randint(12, 18)

    # Increment percentage
    increment_percent = random.uniform(0.05, 0.18)

    month_counter = 0

    current_salary = original_salary

    for year in years:

        for month in months:

            month_counter += 1

            # APPLY SALARY INCREMENT

            if month_counter == increment_month:

                current_salary = current_salary * (
                    1 + increment_percent
                )

            # OVERTIME PAY LOGIC

            # Employees with OverTime = Yes
            # receive overtime more frequently

            if overtime_flag == "Yes":

                overtime_probability = 0.75

            else:

                overtime_probability = 0.10

            if random.random() < overtime_probability:

                overtime_pay = np.random.normal(
                    loc=current_salary * 0.08,
                    scale=current_salary * 0.03
                )

                overtime_pay = max(0, overtime_pay)

            else:

                overtime_pay = 0

            # BONUS LOGIC
            # Linked to PerformanceRating

            # IBM PerformanceRating typically:
            # 3 = Good
            # 4 = Excellent

            if performance == 4:

                bonus_percent = random.uniform(0.10, 0.22)

            elif performance == 3:

                bonus_percent = random.uniform(0.04, 0.10)

            else:

                bonus_percent = random.uniform(0.01, 0.03)

            # Bonus typically paid quarterly
            bonus_months = [3, 6, 9, 12]

            if month in bonus_months:

                bonus = current_salary * bonus_percent

            else:

                bonus = 0

            # TOTAL COMPENSATION

            total_compensation = (
                current_salary
                + overtime_pay
                + bonus
            )

            # APPEND RECORD

            payroll_rows.append({

                "EmployeeNumber": employee_id,

                "EmployeeName": employee_name,

                "Month": month,

                "Year": year,

                "BaseSalary": round(current_salary, 2),

                "OvertimePay": round(overtime_pay, 2),

                "Bonus": round(bonus, 2),

                "TotalCompensation": round(
                    total_compensation, 2
                )
            })

In [13]:
# Create DataFrame from generated payroll data

payroll_df = pd.DataFrame(payroll_rows)

# Sort by EmployeeNumber, Year, Month for better readability

payroll_df = payroll_df.sort_values(
    by=["EmployeeNumber", "Year", "Month"]
)

In [18]:
# Export to CSV
payroll_df.to_csv(
    "Generated_Payroll_24_Months.csv",
    index=False
)

In [21]:
# Output summary of generated dataset
print("Synthetic Payroll Dataset Generated:")
print(f"Employees: {payroll_df['EmployeeNumber'].nunique()}")
print(f"Rows Generated: {len(payroll_df)}")

print("\nColumns:")
print(payroll_df.columns.tolist())

print("\nSample Data:")
print(payroll_df.head())

print("\nCSV SAVED:")
print("Generated_Payroll_24_Months.csv")

Synthetic Payroll Dataset Generated:
Employees: 1470
Rows Generated: 35280

Columns:
['EmployeeNumber', 'EmployeeName', 'Month', 'Year', 'BaseSalary', 'OvertimePay', 'Bonus', 'TotalCompensation']

Sample Data:
   EmployeeNumber  EmployeeName  Month  Year  BaseSalary  OvertimePay   Bonus  \
0               1  Allison Hill      1  2024      5993.0       568.74    0.00   
1               1  Allison Hill      2  2024      5993.0       454.58    0.00   
2               1  Allison Hill      3  2024      5993.0       595.89  435.82   
3               1  Allison Hill      4  2024      5993.0       753.27    0.00   
4               1  Allison Hill      5  2024      5993.0       437.34    0.00   

   TotalCompensation  
0            6561.74  
1            6447.58  
2            7024.71  
3            6746.27  
4            6430.34  

CSV SAVED:
Generated_Payroll_24_Months.csv
